# Figuring out BCPNN

Importo tutti i moduli che abbiamo nella repo:

In [ ]:
import sys
sys.path.insert(0,'..')

In [ ]:
from vigipy import *
import pandas as pd
from pyarrow.parquet import ParquetFile
import pyarrow as pa 

Read the data and only take what we need for the contingency table. Only get first few lines bc data is large

In [ ]:
n=100000

pf = ParquetFile(r"C:\Users\Admin\drug-safety-signal-detection\data\faers_flat_deduped.parquet") 
first_n_rows = next(pf.iter_batches(batch_size = n)) 
ae_df = pa.Table.from_batches([first_n_rows]).to_pandas() 


Filtro il dataframe nel format che vuole vigipy:

In [ ]:
#['AE', 'name', 'count'] ('date' is optional for longitudinal models)

drug_adverse = ae_df.groupby(["reaction_pt","drug_name"]).size().reset_index(name="count")

drug_adverse["AE"] = drug_adverse["reaction_pt"].astype(str)
drug_adverse["name"] = drug_adverse["drug_name"].astype(str)
drug_adverse["count"] = drug_adverse["count"].astype(int)

drug_adverse = drug_adverse[["AE", "name", "count"]]


Ora dovrebbe essere come lo vuole ui

In [ ]:
drug_adverse.head()

Un sacco di problemi con la funzione che crea la contingency table usata in convert, per cui sovrascrivo con una che funzia x me ora:

In [ ]:
import numpy as np
import vigipy.utils.data_prep as data_prep

def fixed_compute_contingency(data_frame, product_label, count_label, ae_label, margin_threshold):
    data_cont = pd.pivot_table(
        data_frame,
        values=count_label,
        index=product_label,
        columns=ae_label,
        aggfunc="sum",
        fill_value=0,
    )
    data_cont = data_cont.astype(float)
    data_cont.index = pd.Index(data_cont.index.astype(str).tolist())
    data_cont.columns = pd.Index(data_cont.columns.astype(str).tolist())

    # Usa boolean mask invece di np.where per evitare il problema PyArrow
    row_mask = np.sum(data_cont.values, axis=1) < margin_threshold
    col_mask = np.sum(data_cont.values, axis=0) < margin_threshold
    
    drop_rows = data_cont.index[row_mask]
    drop_cols = data_cont.columns[col_mask]
    data_cont = data_cont.drop(drop_rows)
    data_cont = data_cont.drop(drop_cols, axis=1)
    return data_cont

data_prep.compute_contingency = fixed_compute_contingency

La conversione dei dati sembra essere il bottleneck: per 10mila righe è ancora immediato, per 100k ci mette 2 minuti.

In [ ]:
data = convert(drug_adverse)

Funziona! caccio dentro bcpnn

In [ ]:
res=bcpnn(container=data, min_events=3, decision_metric="rank", ranking_statistic="quantile")
res.all_signals.head()
# volendo posso esportare su excel
# res.all_signals.to_excel(r"C:\Users\Admin\drug-safety-signal-detection\results\bcpnn_signals.xlsx", index=False)

Questa è una tabellona con tutti i possibili segnali, vogliamo solo quelli significativi. L'articolosu Iapatinib sets IC>0, IC025>0 (lower limit of 95% CI of CI greater than 0), N>=3. 


Però forse aumenterei il threshold min_counts: andrebbe considerato il contesto di Iapatiniib (era particolarmente nuova nel momento in cui hanno scritto l'articolo?), valutiamo.

Filtro il lower threshold del quantile a >0 tanto è un dataframe pandas, che mi permette anche di creare i dataframeini per intensità del segnale (vedi tabella blu sulla bibbia)

In [ ]:
article_signals=res.all_signals[res.all_signals["quantile"] > 0]


weak_signals=article_signals[article_signals["quantile"]<=1.5]
medium_signals=article_signals[(article_signals["quantile"] > 1.5) & (article_signals["quantile"] <= 3)]
strong_signals=article_signals[article_signals["quantile"] > 3]

In [ ]:
strong_signals.head()

Ok!! funzia e sembra avere senso.

## Volendo posso rendere il modello longitudinale:

Do un occhio alla struttura originale per capire come crearlo:

In [ ]:
ae_df.head()

Di temporale abbiamo data, anno e quartile. non devo convertire al suo container specifico ma mi serve inserire la data in drug_adverse. 

vigipy vuole che passi come stringa l'unità di misura di tempo tra che stanno qua: https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases

In [ ]:
#['AE', 'name', 'count', 'date']

drug_adverse = ae_df.groupby(["reaction_pt","drug_name"]).size().reset_index(name="count")

drug_adverse["AE"] = drug_adverse["reaction_pt"].astype(str)
drug_adverse["name"] = drug_adverse["drug_name"].astype(str)
drug_adverse["count"] = drug_adverse["count"].astype(int)
drug_adverse["date"] = ae_df["receive_year"]

drug_adverse = drug_adverse[["AE", "name", "count", "date"]]


Creo un modello che guarda a AE negli anni (YE):

In [ ]:
LM_years=LongitudinalModel(drug_adverse, time_unit='YE')

e poi ci posso applicare qualsiasi algoritmo come nella situa di prima (2 minuti per 100k righe del parquet)

In [ ]:
LM_years.run(bcpnn, min_events=3, decision_metric="rank", ranking_statistic="quantile")

Cosa ci faccio poi con questo?

In [ ]:
for timestamp, result in LM_years.results:
    print("Signals produced prior to {0}:".format(timestamp))
    print(result.signals.head())

Note that there are two ways of running longitudinal models. One is continuous (via run()) and the other is disjoint (via run_disjoint()). The major difference is that run() assumes you are tracking a continually accumulating signal and that the past influences the present. When running run_disjoint(), the assumption is that the current time slice is the only slice relevant for signal detection.